### Ingestão e Leitura de dados brutos

In [0]:
df_payments = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load("dbfs:/Workspace/Users/jcesarsantana215@gmail.com/Medallion-Architecture/dataset/payments.csv")
)
display(df_payments)

In [0]:
df_customers = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load("dbfs:/Workspace/Users/jcesarsantana215@gmail.com/Medallion-Architecture/dataset/customers.csv")
)
display(df_customers)

In [0]:
df_orders = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load("dbfs:/Workspace/Users/jcesarsantana215@gmail.com/Medallion-Architecture/dataset/orders.csv")
)
display(df_orders)

In [0]:
df_order_items = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load("dbfs:/Workspace/Users/jcesarsantana215@gmail.com/Medallion-Architecture/dataset/order_items.csv")
)
display(df_order_items)

In [0]:
df_products = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load("dbfs:/Workspace/Users/jcesarsantana215@gmail.com/Medallion-Architecture/dataset/products.csv")
)
display(df_products)

In [0]:
df_reviews = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load("dbfs:/Workspace/Users/jcesarsantana215@gmail.com/Medallion-Architecture/dataset/reviews.csv")
)
display(df_reviews)

### Persistência na Camada Bronze

Agora que todos os CSVs foram lidos, vamos persistir os dados como tabelas Delta na camada Bronze. Isso garante:
- Formato otimizado (Delta Lake com transações ACID)
- Versionamento e histórico de alterações
- Disponibilidade para as camadas Silver e Gold

In [0]:
from pyspark.sql.functions import current_timestamp

# Criar schema (database) da camada Bronze se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

# Dicionário com todos os DataFrames e seus nomes de tabela
tables = {
    "payments": df_payments,
    "customers": df_customers,
    "orders": df_orders,
    "order_items": df_order_items,
    "products": df_products,
    "reviews": df_reviews,
}

# Salvar cada DataFrame como tabela Delta gerenciada na camada Bronze
for table_name, df in tables.items():
    full_table_name = f"bronze.{table_name}"
    
    (
        df
        .withColumn("_ingested_at", current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(full_table_name)
    )
    print(f"✓ Tabela {full_table_name} salva com sucesso!")
    print(f"  Linhas: {df.count()} | Colunas: {len(df.columns)}")

print("\n✅ Camada Bronze concluída! Todas as tabelas foram persistidas em formato Delta.")